# Spot-check: raw responses and parse results

Verify that the parser is classifying responses correctly by inspecting
raw model output alongside parsed fields.

In [1]:
import json
import sys
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path.cwd().parent))

from src.config import all_cells, raw_output_path, Q13_CANONICAL_CARDS
from src.parser import parse_response


def load_cell(experiment: str, condition: str) -> list[dict]:
    """Load raw JSONL and attach parse results."""
    path = raw_output_path(experiment, condition)
    rows = []
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            parsed = parse_response(experiment, condition, row["text"] or "")
            row["parsed"] = parsed
            rows.append(row)
    return rows

In [2]:
for exp_cond in ["q4_condition0", "q5_condition0"]:
    path = Path(f"data/raw/{exp_cond}.jsonl")
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 3: break
            record = json.loads(line)
            print(f"--- {exp_cond} sample {i} ---")
            print(record.get("text", record))
            print()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/q4_condition0.jsonl'

## 1. Summary table — all 16 cells

In [3]:
print(f"{'Cell':10s} {'Parsed':>7s} {'Outcome':>12s}  Choice distribution")
print("-" * 75)

for exp, cond in all_cells():
    rows = load_cell(exp, cond)
    n = len(rows)
    n_ok = sum(1 for r in rows if r["parsed"]["parsed_ok"])
    cell = f"{exp}_{cond}"

    if exp in ("Q1", "Q4", "Q5"):
        ok = [r["parsed"] for r in rows if r["parsed"]["parsed_ok"]]
        hl = sum(1 for r in ok if r["human_like"])
        choices = Counter((r["scenario_a_choice"], r["scenario_b_choice"]) for r in ok)
        print(f"{cell:10s} {n_ok:3d}/{n:<3d}  hl={hl:2d}/{n_ok:<2d}     {dict(choices)}")
    else:
        ok = [r["parsed"] for r in rows if r["parsed"]["parsed_ok"]]
        c = sum(1 for r in ok if r["correct"])
        h = sum(1 for r in ok if r["human_like_error"])
        o = sum(1 for r in ok if r["other_error"])
        print(f"{cell:10s} {n_ok:3d}/{n:<3d}  c={c:2d} hle={h:2d} o={o:2d}  {Counter(tuple(r['cards_selected']) for r in ok)}")

Cell        Parsed      Outcome  Choice distribution
---------------------------------------------------------------------------
Q1_0        50/50   hl=50/50     {('B', 'A'): 50}
Q1_A        50/50   hl=50/50     {('B', 'A'): 50}
Q1_B        50/50   hl=48/50     {('B', 'A'): 48, ('B', 'B'): 2}
Q1_C        50/50   hl=50/50     {('the certain amount', 'the gamble'): 50}
Q4_0        50/50   hl= 0/50     {('YES', 'YES'): 50}
Q4_A        50/50   hl= 0/50     {('YES', 'YES'): 50}
Q4_B        50/50   hl= 0/50     {('YES', 'YES'): 50}
Q4_C        50/50   hl= 0/50     {('make the trip', 'make the trip'): 50}
Q5_0        50/50   hl= 0/50     {('B', 'B'): 50}
Q5_A        50/50   hl= 0/50     {('B', 'B'): 50}
Q5_B        50/50   hl= 0/50     {('the later payment', 'the later payment'): 50}
Q5_C        50/50   hl= 0/50     {('the later payment', 'the later payment'): 50}
Q13_0       50/50   c=50 hle= 0 o= 0  Counter({('7', 'E'): 50})
Q13_A       50/50   c=50 hle= 0 o= 0  Counter({('3', 'U'): 50})
Q1

## 2. Spot-check: read raw JSON for specific cells

Change `CELL` below to inspect any `(experiment, condition)` pair.
Shows the first N raw model responses with their parsed outcomes.

In [4]:
# ── Change these to inspect different cells ──
CELL_EXP = "Q1"
CELL_COND = "B"   # the one with 48/50
N_SHOW = 5

rows = load_cell(CELL_EXP, CELL_COND)

for row in rows[:N_SHOW]:
    p = row["parsed"]
    print(f"=== {row['custom_id']} ===")
    print(f"  parsed_ok:  {p['parsed_ok']}")
    if CELL_EXP in ("Q1", "Q4", "Q5"):
        print(f"  A choice:   {p['scenario_a_choice']}")
        print(f"  B choice:   {p['scenario_b_choice']}")
        print(f"  human_like: {p['human_like']}")
    else:
        print(f"  cards:      {p['cards_selected']}")
        print(f"  correct:    {p['correct']}")
        print(f"  hle:        {p['human_like_error']}")
    print(f"  --- raw text (first 500 chars) ---")
    print(f"  {(row['text'] or '')[:500]}")
    print()

=== Q1_B_0000 ===
  parsed_ok:  True
  A choice:   B
  B choice:   A
  human_like: True
  --- raw text (first 500 chars) ---
  ```json
{
  "Scenario A": {
    "Choice": "B",
    "Confidence": 0.75,
    "Explanation": "The expected value of option A is $470, which is less than the guaranteed $500 from option B. Taking the sure $500 is both analytically superior and avoids the risk of winning nothing.",
    "Reasoning": "B"
  },
  "Scenario B": {
    "Choice": "A",
    "Confidence": 0.62,
    "Explanation": "The expected loss of option A is $470, which is less than the certain $500 loss of option B. Option A offers a 53% 

=== Q1_B_0001 ===
  parsed_ok:  True
  A choice:   B
  B choice:   A
  human_like: True
  --- raw text (first 500 chars) ---
  ```json
{
  "Scenario A": {
    "Choice": "B",
    "Confidence": 0.85,
    "Explanation": "The expected value of option A is $470, which is less than the guaranteed $500 from option B. Choosing B secures a certain gain and avoids the risk of wi

## 3. Inspect non-human-like responses in Q1_B

Q1_B had 2/50 non-human-like. Show those specifically.

In [9]:
rows = load_cell("Q5", "B")
non_hl = [r for r in rows if r["parsed"]["parsed_ok"] and not r["parsed"]["human_like"]]

print(f"{len(non_hl)} non-human-like responses in Q1_B:\n")
for row in non_hl:
    p = row["parsed"]
    print(f"  {row['custom_id']}: A={p['scenario_a_choice']}, B={p['scenario_b_choice']}")
    print(f"  raw text:")
    print(f"  {(row['text'] or '')[:600]}")
    print()

50 non-human-like responses in Q1_B:

  Q5_B_0000: A=the later payment, B=the later payment
  raw text:
  ```json
{
  "Scenario A": {
    "Choice": "the later payment",
    "Confidence": 0.85,
    "Explanation": "Waiting one day for an extra $10 represents a 10% return overnight, which far exceeds any standard investment return. The minimal delay makes the additional $10 well worth the wait.",
    "Reasoning": "B"
  },
  "Scenario B": {
    "Choice": "the later payment",
    "Confidence": 0.90,
    "Explanation": "The same logic applies: a 10% gain for waiting just one additional day is an exceptional return. Since both payments are already delayed by 30 days, the extra single day is negligible fo

  Q5_B_0001: A=the later payment, B=the later payment
  raw text:
  ```json
{
  "Scenario A": {
    "Choice": "the later payment",
    "Confidence": 0.85,
    "Explanation": "Waiting one day for an extra $10 represents a 10% return overnight, which is an extraordinarily high rate of return. 

## 4. Spot-check Q13_B — verify "83" parsed as a unit

In [6]:
rows = load_cell("Q13", "B")

print(f"Q13_B canonical: {Q13_CANONICAL_CARDS['B']}\n")

# Show first 5 raw Choice values
for row in rows[:5]:
    p = row["parsed"]
    raw_json = p.get("raw_json", {})
    raw_choice = None
    for k, v in raw_json.items():
        if k.strip().upper() == "CHOICE":
            raw_choice = v
            break
    print(f"  {row['custom_id']}")
    print(f"    raw Choice value: {raw_choice!r}")
    print(f"    parsed cards:     {p['cards_selected']}")
    print(f"    correct:          {p['correct']}")
    print()

Q13_B canonical: {'cards': {'Q', '83', 'I', '8'}, 'correct': {'83', 'I'}, 'human_like_error': {'I', '8'}}

  Q13_B_0000
    raw Choice value: 'I and 83'
    parsed cards:     ['83', 'I']
    correct:          True

  Q13_B_0001
    raw Choice value: 'I and 83'
    parsed cards:     ['83', 'I']
    correct:          True

  Q13_B_0002
    raw Choice value: 'I and 83'
    parsed cards:     ['83', 'I']
    correct:          True

  Q13_B_0003
    raw Choice value: 'I and 83'
    parsed cards:     ['83', 'I']
    correct:          True

  Q13_B_0004
    raw Choice value: 'I and 83'
    parsed cards:     ['83', 'I']
    correct:          True



## 5. Random sample — any parse failures?

Should be empty if the summary above showed 800/800 clean.

In [7]:
failures = []
for exp, cond in all_cells():
    for row in load_cell(exp, cond):
        if not row["parsed"]["parsed_ok"]:
            failures.append((f"{exp}_{cond}", row["custom_id"], row["parsed"]["parse_error"], (row["text"] or "")[:200]))

if failures:
    print(f"{len(failures)} PARSE FAILURES:")
    for cell, cid, err, txt in failures:
        print(f"  [{cell}] {cid}: {err}")
        print(f"    {txt!r}\n")
else:
    print("No parse failures across all 800 responses.")

No parse failures across all 800 responses.


## 6. Token usage summary

In [8]:
total_input = 0
total_output = 0

for exp, cond in all_cells():
    path = raw_output_path(exp, cond)
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            usage = row.get("usage")
            if usage:
                total_input += usage["input_tokens"]
                total_output += usage["output_tokens"]

print(f"Total input tokens:  {total_input:,}")
print(f"Total output tokens: {total_output:,}")
print(f"")
# Opus 4.6 batch pricing: $7.50/M input, $37.50/M output
cost_in = total_input / 1_000_000 * 7.50
cost_out = total_output / 1_000_000 * 37.50
print(f"Estimated cost (batch pricing):")
print(f"  Input:  ${cost_in:.2f}")
print(f"  Output: ${cost_out:.2f}")
print(f"  Total:  ${cost_in + cost_out:.2f}")

Total input tokens:  377,150
Total output tokens: 144,137

Estimated cost (batch pricing):
  Input:  $2.83
  Output: $5.41
  Total:  $8.23
